In [ ]:
import sys
import os
sys.path
os.listdir()
os.chdir(os.getcwd().replace("\\","/").replace("/notebooks",""))
sys.path.append("src")

In [ ]:
from src.envConfig import EnvConfig

In [ ]:
EnvConfig()

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
spark = SparkSession.builder \
    .appName("ExcelComPandasSpark4") \
    .config("spark.sql.ansi.enabled", "false") \
    .config("spark.api.python.worker.connection.timeout", "200") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
    .getOrCreate()

In [ ]:
p_df = pd.read_excel("", na_values=["-"])

In [ ]:
df = spark.createDataFrame(p_df)
df.show()

In [ ]:
venda_where = (
    df
    .filter(F.col("Tipo de Movimentação") == "Venda")
)

compra_where = (
    df
    .filter(F.col("Tipo de Movimentação") == "Compra")
)

In [ ]:
venda_select = (
    venda_where
    .select(
        F.col("Data do Negócio"),
        F.col("Código de Negociação"),
        F.col("Quantidade"),
        F.col("Preço"),
        F.col("Valor")
    )
) 

compra_select = (
    compra_where
    .select(
        F.col("Data do Negócio"),
        F.col("Código de Negociação"),
        F.col("Quantidade"),
        F.col("Preço"),
        F.col("Valor")
    )
) 

In [ ]:
from pyspark.sql.functions import to_date
venda_date = (venda_select.withColumn("Data do Negócio", to_date("Data do Negócio", "dd/MM/yyyy")))
compra_date = (compra_select.withColumn("Data do Negócio", to_date("Data do Negócio", "dd/MM/yyyy")))

In [ ]:
venda_order = (
    venda_date
    .orderBy(F.desc("Código de Negociação"), F.desc("Data do Negócio"))
)
compra_order = (
    compra_date
    .orderBy(F.desc("Código de Negociação"), F.desc("Data do Negócio"))
)

In [ ]:
compra_order.show(truncate=False)

In [ ]:
def somar(df):
    result = (
        df
            .groupby('Código de Negociação')
            .agg(
                F.sum("Valor").alias("Valor"),
                F.sum('Quantidade').alias("Quantidade")
            )
            .select(
                F.col('Código de Negociação'),
                F.col('Quantidade'),
                F.col('Valor')
            )
        )
    return result

In [ ]:
compra_df = somar(compra_order)
venda_df = somar(venda_order)

In [ ]:
compra_df.show(truncate=False)

In [ ]:
venda_df.show(truncate=False)